In [0]:
%sql
select current_catalog(), current_schema();

current_catalog(),current_schema()
main,default


In [0]:
# Databricks notebook source
# THEME PARK OPERATIONS - DATASET SETUP
# Upload the five data files to the same Unity Catalog Volume folder first.
# Change BASE_PATH, CATALOG, and SCHEMA below to match your workspace.
# WARNING: Rerunning this notebook overwrites the initial training tables.
# Do NOT rerun it after starting the streaming pipeline.

# COMMAND ----------
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DoubleType)

CATALOG = "main"      # Change to your catalog
SCHEMA = "default"    # Change to your schema
BASE_PATH = "/Workspace/Users/dharshinis369@gmail.com/Assessment1"  # Change to your folder


prefix = f"{CATALOG}.{SCHEMA}"

entry_schema = StructType([
    StructField("entry_id", IntegerType(), True),
    StructField("visitor_id", StringType(), True),
    StructField("attraction_id", StringType(), True),
    StructField("visit_date", StringType(), True),
    StructField("entry_time", StringType(), True),
    StructField("wait_minutes", IntegerType(), True),
    StructField("ride_minutes", IntegerType(), True),
    StructField("ticket_type", StringType(), True),
    StructField("ticket_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("booking_channel", StringType(), True)
])

maintenance_schema = StructType([
    StructField("maintenance_id", StringType(), True),
    StructField("attraction_id", StringType(), True),
    StructField("maintenance_date", StringType(), True),
    StructField("issue_type", StringType(), True),
    StructField("downtime_minutes", IntegerType(), True),
    StructField("maintenance_status", StringType(), True)
])

# COMMAND ----------
def read_csv(filename, schema):
    return (spark.read.format("csv")
            .option("header", "true")
            .schema(schema)
            .load(f"{BASE_PATH}/{filename}"))

entries = read_csv("park_entries.csv", entry_schema)
attractions = spark.read.json(f"{BASE_PATH}/attractions.json")
maintenance = read_csv("attraction_maintenance.csv", maintenance_schema)
changes = read_csv("park_entry_changes.csv", entry_schema)

# COMMAND ----------
# Source table is append-only for later streaming exercises.
entries.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_entries_raw")
attractions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_attractions")
maintenance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_maintenance")
changes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_entry_changes")

# COMMAND ----------
# Inspect the tables. Keep this source unchanged until incremental loading.
for name in ["tp_entries_raw", "tp_attractions", "tp_maintenance", "tp_entry_changes"]:
    print(f"{prefix}.{name}: {spark.table(f'{prefix}.{name}').count()} rows")

display(spark.table(f"{prefix}.tp_entries_raw").limit(10))

# COMMAND ----------
# LATER, after your streaming pipeline has been created and run once:
# Read park_entries_incremental.csv with entry_schema and append it to
# tp_entries_raw. This is intentionally left for the participant to implement.

main.default.tp_entries_raw: 121 rows
main.default.tp_attractions: 10 rows
main.default.tp_maintenance: 28 rows
main.default.tp_entry_changes: 10 rows


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


creating working table for update/delete/merge

A. Data ingestion and exploration

In [0]:
#1
display(spark.table('tp_entries_raw'))
display(spark.table('tp_attractions'))
display(spark.table('tp_maintenance'))
display(spark.table('tp_entry_changes'))



entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


attraction_id,attraction_name,capacity_per_hour,category,features,is_active,location,min_height_cm,opened_year
A01,Sky Spinner,140,Thrill,"List(Outdoor, High Speed)",true,"List(N1, North)",120,2018
A02,Coral Carousel,200,Family,"List(Indoor, Family Friendly)",true,"List(E2, East)",0,2015
A03,Jungle Rapids,110,Water,"List(Water Ride, Outdoor)",true,"List(S1, South)",110,2020
A04,Moonlight Train,180,Family,"List(Scenic, Family Friendly)",true,"List(W3, West)",0,2012
A05,Meteor Drop,100,Thrill,"List(High Speed, Outdoor)",true,"List(N2, North)",135,2022
A06,Pirate Voyage,130,Water,"List(Water Ride, Themed)",true,"List(E1, East)",100,2019
A07,Crystal Maze,160,Adventure,"List(Indoor, Puzzle)",true,"List(S3, South)",90,2021
A08,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A09,Galaxy Theater,260,Show,"List(Indoor, Live Show)",true,"List(N3, North)",0,2017
A10,Mini Explorers,190,Family,"List(Kids, Indoor)",true,"List(E3, East)",0,2016


maintenance_id,attraction_id,maintenance_date,issue_type,downtime_minutes,maintenance_status
M001,A05,2026-10-15,Cleaning,90,Scheduled
M002,A08,2026-10-05,Mechanical,60,Completed
M003,A02,2026-10-12,Mechanical,0,Scheduled
M004,A09,2026-10-18,Electrical,0,Open
M005,A05,2026-10-12,Electrical,0,Scheduled
M006,A04,2026-10-07,Water System,45,Completed
M007,A10,2026-10-20,Mechanical,120,Completed
M008,A04,2026-10-04,Water System,90,Open
M009,A08,2026-10-19,Safety Check,30,Scheduled
M010,A02,2026-10-15,Safety Check,0,Open


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


In [0]:
#2
spark.table('tp_entries_raw').printSchema()

root
 |-- entry_id: integer (nullable = true)
 |-- visitor_id: string (nullable = true)
 |-- attraction_id: string (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- entry_time: string (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- ride_minutes: integer (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- ticket_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- booking_channel: string (nullable = true)



In [0]:
#3
spark.table('tp_entries_raw').select(
    'entry_id',
    'attraction_id',
    'visit_date',
    'ticket_type',
    'ticket_price'
).show(20)

+--------+-------------+----------+-----------+------------+
|entry_id|attraction_id|visit_date|ticket_type|ticket_price|
+--------+-------------+----------+-----------+------------+
|    1001|          A08|2026-10-06|   Standard|       350.0|
|    1002|          A01|2026-10-10|   Standard|       350.0|
|    1003|          A04|2026-10-11|   Standard|       350.0|
|    1004|          A05|2026-10-15|    Express|       650.0|
|    1005|          A07|2026-10-07|   Standard|       350.0|
|    1006|          A08|2026-10-09|        VIP|       900.0|
|    1007|          A08|2026-10-14|   Standard|       350.0|
|    1008|          A06|2026-10-01|    Express|       650.0|
|    1009|          A02|2026-10-16|   Standard|       350.0|
|    1010|          A02|2026-10-09|    Express|       600.0|
|    1011|          A02|2026-10-06|   Standard|       350.0|
|    1012|          A07|2026-10-13|    Express|       600.0|
|    1013|          A09|2026-10-11|   Standard|       350.0|
|    1014|          A07|

In [0]:
%sql
--4
select *
from tp_entries_raw
where status = 'Completed'
and ticket_price > 500;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1014,V564,A07,2026-10-01,14:10:00,65,4,VIP,900.0,Completed,Partner
1018,V535,A10,2026-10-01,13:10:00,65,8,Express,600.0,Completed,App
1021,V527,A03,2026-10-02,13:00:00,-8,8,Express,650.0,Completed,Web
1022,V543,A09,2026-10-15,17:20:00,65,4,Express,600.0,Completed,App
1024,V537,A06,2026-10-06,17:30:00,50,8,Express,650.0,Completed,Web
1025,V513,A09,2026-10-08,14:20:00,65,12,Express,600.0,Completed,App
1028,V547,A08,2026-10-02,13:20:00,75,12,Express,600.0,Completed,App


In [0]:
%sql
--5
select distinct ticket_type 
from tp_entries_raw
where booking_channel in ('App','Partner');


ticket_type
Standard
Express
VIP


In [0]:
%sql
--6
select 
    cast(visit_date as date) as visit_date,
    count(*) as total
from tp_entries_raw
group by cast(visit_date as date);

visit_date,total
2026-10-06,6
2026-10-10,7
2026-10-11,9
2026-10-15,8
2026-10-07,8
2026-10-09,10
2026-10-14,7
2026-10-01,11
2026-10-16,8
2026-10-13,6


B. Cleaning and transforming with PySpark

In [0]:
#7
from pyspark.sql.functions import col,when
spark.table('tp_entries_raw').filter(
    (col('ticket_price').isNull()) |
    (col('booking_channel').isNull())
).show()

+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+
|entry_id|visitor_id|attraction_id|visit_date|entry_time|wait_minutes|ride_minutes|ticket_type|ticket_price|   status|booking_channel|
+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+
|    1009|      V510|          A02|2026-10-16|  16:00:00|          10|           8|   Standard|       350.0|Completed|           NULL|
|    1016|      V563|          A10|2026-10-08|  15:10:00|          30|          10|   Standard|        NULL|Completed|          Kiosk|
|    1040|      V514|          A03|2026-10-01|  19:10:00|           5|          15|   Standard|       400.0|Completed|           NULL|
|    1071|      V553|          A07|2026-10-03|  11:10:00|           5|          20|   Standard|        NULL|Completed|        Partner|
|    1087|      V537|          A01|2026-10-11|  10:00:0

In [0]:
#8
cleaing_df = spark.table('tp_entries_raw').fillna(
    {
        'booking_channel': 'Unknown'
    }
)
cleaing_df.show()


+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+
|entry_id|visitor_id|attraction_id|visit_date|entry_time|wait_minutes|ride_minutes|ticket_type|ticket_price|   status|booking_channel|
+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+
|    1001|      V515|          A08|2026-10-06|  17:20:00|          15|          20|   Standard|       350.0|Cancelled|            App|
|    1002|      V535|          A01|2026-10-10|  15:00:00|          65|           6|   Standard|       350.0|Completed|            App|
|    1003|      V547|          A04|2026-10-11|  11:00:00|          40|          15|   Standard|       350.0|Completed|            Web|
|    1004|      V517|          A05|2026-10-15|  17:50:00|          20|           6|    Express|       650.0|Completed|            Web|
|    1005|      V537|          A07|2026-10-07|  16:50:0

In [0]:
#9
from pyspark.sql.functions import col
spark.table('tp_entries_raw').filter(
    (col('wait_minutes') < 0) |
    (col('ride_minutes')==0) |
    (col('ticket_price') <= 0)
).show()

+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+
|entry_id|visitor_id|attraction_id|visit_date|entry_time|wait_minutes|ride_minutes|ticket_type|ticket_price|   status|booking_channel|
+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+
|    1021|      V527|          A03|2026-10-02|  13:00:00|          -8|           8|    Express|       650.0|Completed|            Web|
|    1036|      V527|          A06|2026-10-04|  14:40:00|          40|           0|   Standard|       400.0|Completed|          Kiosk|
|    1058|      V516|          A02|2026-10-08|  19:10:00|          30|           4|   Standard|         0.0|Completed|            App|
|    1073|      V501|          A03|2026-10-07|  18:50:00|          -3|           6|    Express|       650.0|Completed|            App|
+--------+----------+-------------+----------+---------

In [0]:
#10
from pyspark.sql.functions import count
spark.table('tp_entries_raw').groupby('entry_id').agg(
    count('*').alias('count')
).filter(
    col('count') > 1
).show()

deduplicated_df = spark.table('tp_entries_raw').dropDuplicates(['entry_id'])
display(deduplicated_df)

+--------+-----+
|entry_id|count|
+--------+-----+
|    1032|    2|
+--------+-----+



entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
#11
entries = spark.table('tp_entries_raw').alias('e')
attractions = spark.table('tp_attractions').alias('a')
unmatched_df = entries.join(
    attractions,
    on='attraction_id',
    how='left_anti'
)
display(unmatched_df)


attraction_id,entry_id,visitor_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
A99,1092,V566,2026-10-03,15:40:00,75,20,Standard,400.0,Completed,Kiosk


In [0]:
#12
from pyspark.sql.functions import when
entry_df = spark.table('tp_entries_raw').dropna(subset=['wait_minutes']).withColumn(
    'wait_category',
    when(
        (col('wait_minutes')>=0) &
        (col('wait_minutes')<=20), 'Short'
    )
    .when(
        (col('wait_minutes')>=21) &
        (col('wait_minutes')<=45), 'Medium'
    )
    .when(
        col('wait_minutes') > 45 , 'Long'
    )
)
entry_df.show()

+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+-------------+
|entry_id|visitor_id|attraction_id|visit_date|entry_time|wait_minutes|ride_minutes|ticket_type|ticket_price|   status|booking_channel|wait_category|
+--------+----------+-------------+----------+----------+------------+------------+-----------+------------+---------+---------------+-------------+
|    1001|      V515|          A08|2026-10-06|  17:20:00|          15|          20|   Standard|       350.0|Cancelled|            App|        Short|
|    1002|      V535|          A01|2026-10-10|  15:00:00|          65|           6|   Standard|       350.0|Completed|            App|         Long|
|    1003|      V547|          A04|2026-10-11|  11:00:00|          40|          15|   Standard|       350.0|Completed|            Web|       Medium|
|    1004|      V517|          A05|2026-10-15|  17:50:00|          20|           6|    Express|       650.

In [0]:
#13
joining_entries_df = entry_df.join(
    spark.table('tp_attractions'),
    on='attraction_id',
    how='inner'
)
joining_entries_df.select(
    col('location.zone'),
    col('features')
).show()

+-----+--------------------+
| zone|            features|
+-----+--------------------+
| West|[Outdoor, Interac...|
|North|[Outdoor, High Sp...|
| West|[Scenic, Family F...|
|North|[High Speed, Outd...|
|South|    [Indoor, Puzzle]|
| West|[Outdoor, Interac...|
| West|[Outdoor, Interac...|
| East|[Water Ride, Themed]|
| East|[Indoor, Family F...|
| East|[Indoor, Family F...|
| East|[Indoor, Family F...|
|South|    [Indoor, Puzzle]|
|North| [Indoor, Live Show]|
|South|    [Indoor, Puzzle]|
|North|[High Speed, Outd...|
| East|      [Kids, Indoor]|
| East|      [Kids, Indoor]|
| East|      [Kids, Indoor]|
| East|[Water Ride, Themed]|
| East|[Water Ride, Themed]|
+-----+--------------------+
only showing top 20 rows


C. SQL joins and aggregations

In [0]:
%sql
--14
select *
from tp_entries_raw
where status = 'Completed'
and ticket_type in ('Standard','Express')
and ticket_price between 200 and 500;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1013,V506,A09,2026-10-11,10:20:00,15,20,Standard,350.0,Completed,Web
1015,V529,A05,2026-10-01,13:40:00,10,15,Standard,400.0,Completed,Kiosk
1017,V501,A10,2026-10-06,17:50:00,40,8,Standard,350.0,Completed,Partner
1019,V551,A06,2026-10-12,12:10:00,15,6,Standard,400.0,Completed,App
1026,V512,A04,2026-10-15,15:30:00,20,8,Standard,350.0,Completed,Kiosk


In [0]:
%sql
--15
select 
    attraction_id,
    sum(case when ticket_price > 0 then ticket_price else 0 end) as total_revenue,
    count(*) as completed_entry_count
from tp_entries_raw
where status = 'Completed'
group by attraction_id;

attraction_id,total_revenue,completed_entry_count
A01,4200.0,9
A04,5450.0,12
A05,5850.0,10
A07,4750.0,10
A08,3600.0,8
A06,4600.0,9
A02,2000.0,6
A09,3200.0,7
A10,6050.0,14
A03,4500.0,8


In [0]:
%sql
--16
select 
    a.category as attraction_category,
    avg(e.wait_minutes) as avg_wait_minutes
from tp_entries_raw e
join tp_attractions a
    on e.attraction_id = a.attraction_id
where e.wait_minutes >= 0
group by a.category;

attraction_category,avg_wait_minutes
Family,31.463414634146343
Thrill,30.5
Show,35.90909090909091
Adventure,33.958333333333336
Water,25.227272727272727


In [0]:
%sql
--17
select 
    a.attraction_id,
    sum(m.downtime_minutes)
from tp_maintenance m 
join tp_attractions a
    on m.attraction_id = a.attraction_id
where m.maintenance_status = 'Completed'
group by a.attraction_id;


attraction_id,sum(m.downtime_minutes)
A10,120
A02,240
A09,0
A01,120
A03,30
A07,120
A06,135
A04,75
A08,60


In [0]:
%sql
--18
select 
    a.location.zone as zone,
    sum(e.ticket_price) as total_revenue
from tp_entries_raw e 
join tp_attractions a
    on e.attraction_id = a.attraction_id
where e.status = 'Completed'
and e.ticket_price > 0
group by a.location.zone;


zone,total_revenue
North,13250.0
East,12650.0
South,9250.0
West,9050.0


D. Delta table creation and CRUD

In [0]:
%sql
--19
create or replace table tp_entries_work
using delta
as
select * from tp_entries_raw;

num_affected_rows,num_inserted_rows


In [0]:
#20
df = spark.table('tp_entries_raw')
df.write.format('delta').mode('overwrite').saveAsTable('tp_entries')

In [0]:
%sql
--21
insert into tp_entries_work
values 
(1121,'V528','A01','2023-01-01','10:00:00',10,10,'Standard',200,'Completed','Web'),
(1122,'V523','A02','2023-01-01','10:00:00',10,10,'Standard',200,'Completed','Web');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
--22
update tp_entries_work
set status = 'Completed',
ticket_price = 450
where entry_id = 1023;

num_affected_rows
1


In [0]:
%sql
--23
delete from tp_entries_work
where entry_id =1020;

num_affected_rows
1


In [0]:
%sql
select *
from tp_entries_work
where entry_id =1020;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel


In [0]:
%sql
--24
describe extended tp_entries_work;
--managed table: manage both table metadata and data storage
--external table: manage table metadata, data stored separately and managed externally

col_name,data_type,comment
entry_id,int,null
visitor_id,string,null
attraction_id,string,null
visit_date,string,null
entry_time,string,null
wait_minutes,int,null
ride_minutes,int,null
ticket_type,string,null
ticket_price,double,null
status,string,null


E. MERGE / UPSERT and SCD

In [0]:
%sql
--25
select *
from tp_entry_changes c
left join tp_entries_work w  
on c.entry_id = w.entry_id 
where w.entry_id is not null;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App,1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App,1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App,1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,400.0,Completed,null
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App,1068,V531,A10,2026-10-15,16:20:00,30,4,Express,600.0,Completed,Partner
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App,1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,350.0,Pending,Partner
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web,1121,V528,A01,2023-01-01,10:00:00,10,10,Standard,200.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web,1122,V523,A02,2023-01-01,10:00:00,10,10,Standard,200.0,Completed,Web


In [0]:
%sql
--26
merge into tp_entries_work as w 
using tp_entry_changes as c
on c.entry_id = w.entry_id
when matched then
update set  
    w.visitor_id = c.visitor_id,
    w.attraction_id = c.attraction_id,
    w.visit_date = c.visit_date,
    w.entry_time = c.entry_time,
    w.wait_minutes = c.wait_minutes,
    w.ride_minutes = c.ride_minutes,
    w.ticket_type = c.ticket_type,
    w.ticket_price = c.ticket_price,
    w.status = c.status,
    w.booking_channel = c.booking_channel
when not matched then
insert (
    entry_id,
    visitor_id,
    attraction_id,
    visit_date,
    entry_time,
    wait_minutes,
    ride_minutes,
    ticket_type,
    ticket_price,
    status,
    booking_channel
)
values (
    c.entry_id,
    c.visitor_id,
    c.attraction_id,
    c.visit_date,
    c.entry_time,
    c.wait_minutes,
    c.ride_minutes,
    c.ticket_type,
    c.ticket_price,
    c.status,
    c.booking_channel
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10,7,0,3


In [0]:
%sql
describe history tp_entries_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-11-11T06:28:45.000Z,1316548390739548,dharshinis369@gmail.com,MERGE,"Map(predicate -> [""(entry_id#24742 = entry_id#24731)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(750623490769570),141a3391-90cb-44d1-902c-5a314208802b,1111-040250-a16h589t-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3575, numTargetBytesRemoved -> 3169, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 7, executionTimeMs -> 3209, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 1023, numTargetRowsUpdated -> 7, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2103)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-11T06:15:36.000Z,1316548390739548,dharshinis369@gmail.com,DELETE,"Map(predicate -> [""(entry_id#23000 = 1020)""])",null,List(750623490769570),92a7d926-a64c-4f86-a4be-01039e6a9c60,1111-040250-a16h589t-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1037, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 568, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 468)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-11T06:14:26.000Z,1316548390739548,dharshinis369@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#22293 = 1023)""])",null,List(750623490769570),a40e082e-9232-4061-9ae5-34ffc0e419e4,1111-040250-a16h589t-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3667, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1863, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3094, rewriteTimeMs -> 1755)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-11T06:11:51.000Z,1316548390739548,dharshinis369@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(750623490769570),2ddefd70-ff87-4770-b95a-7d30f326b459,1111-040250-a16h589t-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 3169)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T06:06:31.000Z,1316548390739548,dharshinis369@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(750623490769570),ae05c181-e261-4a33-865d-b733292475f5,1111-040250-a16h589t-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4632, numDeletionVectorsRemoved -> 0, numOutputRows -> 121, numOutputBytes -> 4632)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T04:17:22.000Z,1316548390739548,dharshinis369@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.com

In [0]:
%sql
--27 
--updated entry IDs
select * from tp_entries_work version as of 4

except 

select * from tp_entries_work version as of 5;
    

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,350.0,Pending,Partner
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,400.0,Completed,null
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,600.0,Completed,Partner
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk
1122,V523,A02,2023-01-01,10:00:00,10,10,Standard,200.0,Completed,Web
1121,V528,A01,2023-01-01,10:00:00,10,10,Standard,200.0,Completed,Web


In [0]:
%sql
--inserted
select *
from tp_entries_work version as of 5 
where entry_id not in (select entry_id from tp_entries_work version as of 4);



entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


In [0]:
#28
from delta.tables import DeltaTable
'''creating working table in pysprk'''
spark.table('tp_entries_raw').write.format('delta').mode('overwrite').saveAsTable('tp_entries_pyspark')
                                                                                  
target = DeltaTable.forName(spark, 'tp_entries_pyspark')
source = spark.table('tp_entry_changes')
target.alias('t').merge(
    source.alias('s'),
    't.entry_id = s.entry_id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()          
display(spark.table('tp_entries_pyspark'))                                 

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1011,V536,A02,2026-10-06,16:40:00,25,8,Standard,350.0,Cancelled,App
1012,V543,A07,2026-10-13,12:20:00,25,20,Express,600.0,Cancelled,Web


In [0]:
%sql
--29
create or replace table attraction_capacity_history
using delta
as
select
    attraction_id,
    attraction_name,
    capacity_per_hour,
    CURRENT_DATE() as valid_from,
    CAST(NULL AS DATE) AS valid_to,
    TRUE AS is_current
FROM tp_attractions;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6384867094548019>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '--29\ncreate table attraction_capacity_history\nusing delta\nas\nselect\n    attraction_id,\n    attraction_name,\n    capacity_per_hour,\n    CURRENT_DATE() as valid_from,\n    CAST(NULL AS DATE) AS valid_to,\n    TRUE AS is_current\nFROM tp_attractions;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MA

F. Delta internals, Time Travel, and optimization

In [0]:
%sql
--31
describe detail tp_entries_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,5a9ade0c-44b0-4838-b72b-cf619ef3ae8a,main.default.tp_entries_work,null,,2026-11-11T06:06:29.652Z,2026-11-11T06:28:45.000Z,List(),List(),3,11301,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 7, numDeletionVectors -> 1)",false


In [0]:
%sql
describe history tp_entries_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-11-11T06:28:45.000Z,1316548390739548,dharshinis369@gmail.com,MERGE,"Map(predicate -> [""(entry_id#24742 = entry_id#24731)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(750623490769570),141a3391-90cb-44d1-902c-5a314208802b,1111-040250-a16h589t-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3575, numTargetBytesRemoved -> 3169, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 7, executionTimeMs -> 3209, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 1023, numTargetRowsUpdated -> 7, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2103)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-11T06:15:36.000Z,1316548390739548,dharshinis369@gmail.com,DELETE,"Map(predicate -> [""(entry_id#23000 = 1020)""])",null,List(750623490769570),92a7d926-a64c-4f86-a4be-01039e6a9c60,1111-040250-a16h589t-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1037, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 568, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 468)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-11T06:14:26.000Z,1316548390739548,dharshinis369@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#22293 = 1023)""])",null,List(750623490769570),a40e082e-9232-4061-9ae5-34ffc0e419e4,1111-040250-a16h589t-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3667, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1863, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3094, rewriteTimeMs -> 1755)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-11T06:11:51.000Z,1316548390739548,dharshinis369@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(750623490769570),2ddefd70-ff87-4770-b95a-7d30f326b459,1111-040250-a16h589t-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 3169)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T06:06:31.000Z,1316548390739548,dharshinis369@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(750623490769570),ae05c181-e261-4a33-865d-b733292475f5,1111-040250-a16h589t-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 4632, numDeletionVectorsRemoved -> 0, numOutputRows -> 121, numOutputBytes -> 4632)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T04:17:22.000Z,1316548390739548,dharshinis369@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.com

In [0]:
%sql
--32
select *
from tp_entries_work version as of 4;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
%sql
--33
select 'Before Merge' as operation, *
from tp_entries_work version as of 4
where entry_id = 1122
union all
select 'After Merge' as operation, *
from tp_entries_work version as of 5
where entry_id = 1122;

operation,entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
Before Merge,1122,V523,A02,2023-01-01,10:00:00,10,10,Standard,200.0,Completed,Web
After Merge,1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web


In [0]:
%sql
--34
optimize tp_entries_work;
--The numFiles before optimize is 3 and now it is 1. This is because the data is compacted into one file which will help to speed up the performance

path,metrics
,"List(1, 3, List(4770, 4770, 4770.0, 1, 4770), List(3094, 4632, 3767.0, 3, 11301), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1794380525672, 1794380528064, 8, 1, null, List(1, 7), null, 11, 11, 522, 0, null, null, 0)"


In [0]:
%sql
--35
optimize tp_entries_work
zorder by (attraction_id);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 4770), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1794380573308, 1794380574009, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null, 0)"


In [0]:
%sql
--36
vacuum tp_entries_work dry run;
--vacuum removes the old or unused data files.The default retention period is 7 days. The time travel will be affected if the unused files got deleted.

path


G. Views and Parquet

In [0]:
%sql
--37
create or replace view valid_completed_entries
as
select er.*, a.attraction_name
from tp_entries_raw er                            
join tp_attractions a
on er.attraction_id = a.attraction_id
where er.status = 'Completed';

In [0]:
%sql
--38
create or replace temporary view temp_view
as
select *
from tp_entries_raw
where wait_minutes > 45;

In [0]:
%sql
--39
create or replace global temporary view global_temp_view
as 
select *
from tp_entries_raw
where status = 'Completed';

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8112013244429760>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "--39\ncreate or replace global temporary view global_temp_view\nas \nselect *\nfrom tp_entries_raw\nwhere status = 'Completed';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(sel

In [0]:
#40
df = spark.table('main.default.tp_entries_raw')
df.write.format('parquet').mode('overwrite').save('/Volumes/workspace/default/demo_volume/tp_entries')
parquet_df = spark.read.format("parquet").load("/Volumes/workspace/default/demo_volume/tp_entries")
parquet_df.printSchema()


root
 |-- entry_id: integer (nullable = true)
 |-- visitor_id: string (nullable = true)
 |-- attraction_id: string (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- entry_time: string (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- ride_minutes: integer (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- ticket_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- booking_channel: string (nullable = true)



In [0]:
#41
parquet_df = spark.read.format('parquet').load('/Volumes/workspace/default/demo_volume/tp_entries')
parquet_df.write.format('delta').mode('overwrite').saveAsTable('main.default.tp_entries_delta')


H. Structured Streaming

In [0]:
#42
from pyspark.sql.functions import col, concat, lit, when

streaming_df = spark.readStream.format('rate').option('rowsPerSecond',5).load()


In [0]:
#43

visitor_df = streaming_df.withColumn(
    'visitor_event_id',
    col('value')+1000
).withColumn(
    'simulated_wait_minutes',
    (col('value')%60)+1
)


In [0]:
#44
filtered_df = visitor_df.filter(
    col('simulated_wait_minutes') > 30
)

In [0]:
#45
from pyspark.sql.functions import when, count
category_df = filtered_df.withColumn(
    'wait_category',
    when(col('simulated_wait_minutes')<=45,'Medium').otherwise('Long')
)
result_df = category_df.groupBy(
    'wait_category'
).agg(
    count('*').alias('total_events')
)

In [0]:
query = result_df.writeStream.outputMode('complete').format('console').option(
        'checkpointLocation',
        '/Volumes/workspace/default/demo_volume/checkpoint_q45'
    ).trigger(availableNow=True).start()

In [0]:
query.stop()

Final capstone project

In [0]:
%sql
create or refresh streaming table bronze_entries
using delta
as
select *
from stream(tp_entries_raw);

message
The operation was successfully executed.


In [0]:
%sql
create or replace streaming table bronze_attractions
using delta
as
select *
from stream(tp_attractions);

message
The operation was successfully executed.


In [0]:
%sql
create or replace streaming table bronze_maintenance
using delta
as
select *
from stream(tp_maintenance);

message
The operation was successfully executed.


In [0]:
%sql
create or replace table silver_entries as
select distinct *
from bronze_entries
where wait_minutes >= 0
and ride_minutes > 0
and ticket_price > 0
and attraction_id is not null;

num_affected_rows,num_inserted_rows


In [0]:
%sql
update silver_entries
set booking_channel = 'Unknown'
where booking_channel is null;

num_affected_rows
3


In [0]:
%sql
create or replace table gold_daily_summary as
select
    cast(visit_date as date) as visit_date,
    count(*) as total_visitors,
    sum(ticket_price) as total_revenue,
    round(avg(wait_minutes), 2) as avg_wait_time
from silver_entries
where status = 'Completed'
group by cast(visit_date as date);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from gold_daily_summary;

visit_date,total_visitors,total_revenue,avg_wait_time
2026-10-09,5,2350.0,25.0
2026-10-14,4,2250.0,31.25
2026-10-10,5,1750.0,36.0
2026-10-01,10,5250.0,43.5
2026-10-02,2,1000.0,47.5
2026-10-07,5,2250.0,28.0
2026-10-16,7,3000.0,31.43
2026-10-04,6,2450.0,38.33
2026-10-13,5,2600.0,27.0
2026-10-03,7,3900.0,28.57
